# Product Experiment Analysis: Cookie Cats A/B Test

## Business Context

Cookie Cats is a mobile puzzle game where players progress through levels. To evaluate the impact of game progression on player retention, the product team tested moving the first progression gate from level 30 to level 40.

As a Product Analyst, the objective is to determine whether this change improved player retention and whether it should be rolled out to all users.

---

## Business Question

Does moving the progression gate from level 30 to level 40 significantly improve player retention, and should Cookie Cats ship this change?

---

## Success Metrics

- Day 1 Retention
- Day 7 Retention
- Player Engagement (sum_gamerounds)

In [2]:
import pandas as pd
import numpy as np

In [3]:
df=pd.read_csv("../data/raw/cookie_cats.csv")

In [7]:
df.head()

,userid,version,sum_gamerounds,retention_1,retention_7
0,116,gate_30,3,False,False
1,337,gate_30,38,True,False
2,377,gate_40,165,True,False
3,483,gate_40,1,False,False
4,488,gate_40,179,True,True


### Interpretation

The dataset appears to have loaded successfully.

Each row represents a unique player who participated in the experiment.

From the preview, we observe the following variables:

- **userid** – Unique identifier assigned to each player.
- **version** – Experimental group (`gate_30` or `gate_40`).
- **sum_gamerounds** – Total number of game rounds played.
- **retention_1** – Whether the player returned after one day.
- **retention_7** – Whether the player returned after seven days.

These variables are sufficient to evaluate the effect of moving the game progression gate on player retention.

# 3. Dataset Dimensions

## Objective

Understanding the size of the dataset is an important first step.

The `shape` attribute returns:

- Number of rows (observations)
- Number of columns (variables)

This helps estimate the scale of the experiment and gives an initial indication of whether the dataset is large enough for reliable statistical analysis.

In [8]:
df.shape

(90189, 5)

### Interpretation

The dataset contains **90,189 observations** and **5 variables**.

Each observation corresponds to one player who participated in the A/B experiment.

A dataset of this size generally provides sufficient statistical power for comparing retention between the two experimental groups, although this assumption will be formally verified later using power analysis.

# 4. Inspecting the Last Records

## Objective

Viewing the last few rows helps verify that the dataset is complete and consistently formatted from beginning to end.

It also allows us to detect obvious truncation or formatting issues that may not appear in the first few rows.

In [9]:
df.tail()

,userid,version,sum_gamerounds,retention_1,retention_7
90184,9999441,gate_40,97,True,False
90185,9999479,gate_40,30,False,False
90186,9999710,gate_30,28,True,False
90187,9999768,gate_40,51,True,False
90188,9999861,gate_40,16,False,False


### Interpretation

The last records follow the same structure as the first records, indicating that the dataset has been loaded correctly without apparent corruption or truncation.

# 5. Understanding the Variables

## Objective

Before beginning any analysis, it is essential to understand what information is available.

Listing the column names provides a quick overview of the variables that will be used throughout the project.

In [10]:
df.columns

Index(['userid', 'version', 'sum_gamerounds', 'retention_1', 'retention_7'], dtype='str')

### Interpretation

The dataset contains five variables:

- **userid**: Unique identifier for each player.
- **version**: Experimental variant assigned to the player.
- **sum_gamerounds**: Total number of game rounds played.
- **retention_1**: Indicates whether the player returned one day after installation.
- **retention_7**: Indicates whether the player returned seven days after installation.

These variables provide both behavioral (game rounds) and outcome (retention) metrics required to evaluate the experiment.

# 6. Understanding Data Types

## Objective

Every variable in a dataset has a specific data type, such as integer, string, boolean, or float. Verifying data types is an important data validation step because many statistical methods and visualizations depend on variables having the correct type.

For example:

- Numerical variables can be used for mathematical calculations.
- Categorical variables are used for grouping and comparison.
- Boolean variables are commonly used in retention and conversion analysis.

Incorrect data types can lead to inaccurate analyses or unexpected errors later in the project.

In [11]:
df.dtypes

userid            int64
version             str
sum_gamerounds    int64
retention_1        bool
retention_7        bool
dtype: object

### Interpretation

The dataset contains three different types of variables:

- **userid** is stored as an integer, which uniquely identifies each player.
- **version** is stored as an object (string), representing the experimental group assigned to the player.
- **sum_gamerounds** is an integer representing player engagement.
- **retention_1** and **retention_7** are boolean variables indicating whether a player returned after 1 day and 7 days, respectively.

The data types are appropriate for the intended analyses and do not require any conversion at this stage.

# 7. Dataset Overview

## Objective

The `info()` function provides a concise summary of the dataset, including:

- Total number of observations.
- Number of non-null values.
- Data type of each variable.
- Approximate memory usage.

This is one of the quickest ways to assess data quality before beginning exploratory analysis.

In [12]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 90189 entries, 0 to 90188
Data columns (total 5 columns):
 #   Column          Non-Null Count  Dtype
---  ------          --------------  -----
 0   userid          90189 non-null  int64
 1   version         90189 non-null  str  
 2   sum_gamerounds  90189 non-null  int64
 3   retention_1     90189 non-null  bool 
 4   retention_7     90189 non-null  bool 
dtypes: bool(2), int64(2), str(1)
memory usage: 2.2 MB


### Interpretation

The dataset contains 90,189 player records across five variables.

Each variable has the expected number of non-null observations, suggesting that there are no immediately visible missing values.

The memory footprint is relatively small, making the dataset efficient to process using pandas.

# 8. Descriptive Statistics

## Objective

Descriptive statistics summarize the numerical characteristics of the dataset.

The `describe()` function provides information such as:

- Count
- Mean
- Standard deviation
- Minimum
- Quartiles (25%, 50%, 75%)
- Maximum

These statistics help identify the typical player behavior, the spread of the data, and potential outliers that may require further investigation.

In [13]:
df.describe()


,userid,sum_gamerounds
count,9.018900e+04,90189.000000
mean,4.998412e+06,51.872457
std,2.883286e+06,195.050858
min,1.160000e+02,0.000000
25%,2.512230e+06,5.000000
50%,4.995815e+06,16.000000
75%,7.496452e+06,51.000000
max,9.999861e+06,49854.000000


### Interpretation

The descriptive statistics reveal important characteristics of player engagement.

### Key Insights

- The experiment includes **90,189 players**, providing a large sample for reliable statistical analysis.
- Players completed an average of **51.87 game rounds** during the observation period.
- The **median engagement is only 16 rounds**, indicating that over half of the players have relatively low engagement.
- The middle 50% of players completed **5 to 51 rounds**, suggesting that most users do not play extensively.
- The maximum value (**49,854 rounds**) is substantially higher than the typical player, indicating the presence of one or more extreme observations.
- The large gap between the **mean (51.87)** and **median (16)** suggests a **positively (right) skewed** distribution of player engagement.

### Business Relevance

- Most players disengage after relatively few game rounds, making early retention a critical business metric.
- Highly engaged players can disproportionately influence average-based metrics, so outlier analysis is required before drawing conclusions.
- The engagement distribution should be examined visually before selecting appropriate statistical methods.

# 9. Missing Value Analysis

## Objective

Missing values can introduce bias and affect the validity of statistical analyses.

Before evaluating the experiment, it is important to verify whether any observations are incomplete and determine if data cleaning is required.


In [4]:
df.isnull().sum()

userid            0
version           0
sum_gamerounds    0
retention_1       0
retention_7       0
dtype: int64

### Key Insights

- No missing values were identified across any of the variables.
- Every player record contains complete information required for analysis.
- No data imputation or record removal is necessary.

### Business Relevance

- Complete data improves the reliability of statistical testing.
- The experiment can be analyzed without concerns about missing observations affecting the results.

# 10. Duplicate User Validation

## Objective

Each player should appear only once in the experiment.

Duplicate user IDs may indicate data collection errors and could bias retention estimates by giving certain players more influence than others.


In [5]:
df["userid"].duplicated().sum()

np.int64(0)

### Key Insights

- No duplicate user IDs were detected.
- Each observation represents a unique player.
- The experimental unit is correctly maintained.

### Business Relevance

- Statistical tests assume independent observations.
- Unique player records ensure that each participant contributes equally to the experiment.

# 11. Unique Values

## Objective

Understanding the distinct values of each variable helps verify that the dataset matches the expected experimental design.

This step is particularly important for identifying categorical variables and validating the treatment groups.

In [6]:
for col in df.columns:
    print(f"\n{col}")
    print(df[col].unique())


userid
[    116     337     377 ... 9999710 9999768 9999861]

version
<StringArray>
['gate_30', 'gate_40']
Length: 2, dtype: str

sum_gamerounds
[    3    38   165     1   179   187     0     2   108   153    30    39
   305    73    14   204     9     5    45   105     4    80    53    37
    13    50   202    23   103    27    15    49     7   386    25    12
     6    29   136    10    76   138    94    26   127    21   297    33
   139    19    16    95     8   106    22    11   246    54    66   122
    75    17    24    47    31    34    56    60    67    42    65   129
    51    46   104    36    20    74    72    18    79    99    91   149
    68   277   493   274    32    85   191   115   134   152   100   250
   147   391    61    57    35    77   197    69   260    48   270    59
   141   467    40    43    93   222   334    58    84    41   303   177
   216   227    87    71   567    55    78   552    96   107    28   113
   339   452   148   163   220    63   290    86   

### Key Insights

- The experiment contains exactly two treatment groups: **gate_30** and **gate_40**.
- Retention variables are binary and contain only **True** and **False** values.
- User IDs are unique identifiers.
- Game rounds represent continuous engagement values.

### Business Relevance

- The dataset structure aligns with the expected A/B experiment design.
- No unexpected categories or invalid values are immediately observed.

| Variable       | Description              | Data Type   | Business Importance       |
| -------------- | ------------------------ | ----------- | ------------------------- |
| userid         | Unique player identifier | Integer     | Used to verify uniqueness |
| version        | Experimental variant     | Categorical | Independent variable      |
| sum_gamerounds | Total rounds played      | Numeric     | Engagement metric         |
| retention_1    | Returned after Day 1     | Boolean     | Primary KPI               |
| retention_7    | Returned after Day 7     | Boolean     | Long-term KPI             |
